# CDKN2A Analysis

**What this produces:**
1. Cell-type-annotated UMAP
2. p14ARF / p16INK4a feature plots on that UMAP
3. A p14/p16 × cell-type heatmap (mean expression)
4. A detection-rate bar chart by cell type 
5. A saved summary table per dataset, so results can be compared across datasets


Section 0- Imports and Colours  

Section 1- Configuration 


## 0. Imports and colour palette

Replaces scanpy's default continuous colormap (viridis: purple→yellow) with a
grey (zero) → single-colour ramp. Zero-expressing cells read as neutral grey instead of dark purple, p14/p16 keep the project's green/blue convention.

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path


In [ ]:
sc.settings.verbosity = 1
sc.set_figure_params(dpi=100, facecolor="white", frameon=False)

def clear_cmap(hex_color, name="clear_cmap"):
    """Grey (zero) -> hex_color (max) sequential colormap, replaces viridis."""
    return mcolors.LinearSegmentedColormap.from_list(name, ["#E6E6E6", hex_color], N=256)

P14_COLOR = "#3C8D5B"     # green
P16_COLOR = "#2E6FBE"     # blue
CMAP_P14 = clear_cmap(P14_COLOR)
CMAP_P16 = clear_cmap(P16_COLOR)

# fixed categorical palette for cell types/clusters - stays legible past ~10 categories,
# unlike scanpy's default which recycles hues
CATEGORICAL_PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
    "#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E",
    "#E6AB02", "#A6761D", "#666666", "#1F78B4", "#B2DF8A",
    "#FB9A99", "#FDBF6F", "#CAB2D6", "#FFFF99", ]

## 1. Load the matrix

In [ ]:
DATASET_NAME = ""     

DATA_DIR = Path()
CLASSIFICATION_FILENAME = "ccRCC2_classification_classification.txt"

OUTPUT_DIR = DATA_DIR / "analysis_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

P14_ENST = "ENST00000579755"
P16_ENST = "ENST00000304494"

In [ ]:
mtx_path = DATA_DIR
classification_path = DATA_DIR / CLASSIFICATION_FILENAME

print("Files in matrix folder:")
for f in sorted(os.listdir(mtx_path)):
    print(" ", f)

adata = sc.read_10x_mtx(mtx_path, var_names="gene_symbols", cache=False)
adata.var_names_make_unique()
print()
print(adata)

In [ ]:
classification = pd.read_csv(classification_path, sep="\t")
print(classification.shape)
print(classification.columns.tolist())

This cell opens the pigeon classification file (a tab-separated text file) and loads it into a table called classification, where each row is one isoform and the columns describe it (which gene it belongs to, which known transcript it matches, its structural category, etc.).

## 2. Locate the CDKN2A isoforms

The matrix stores each transcript separately, so CDKN2A appears as several rows
(`CDKN2A`, `CDKN2A-1`, `CDKN2A-2`, …). This cell lists them alongside their PacBio IDs.
Note `CDKN2AIP` is a **different gene** — the `-` suffix filter below (`CDKN2A-`) avoids
it, but here we show everything containing "CDKN2A" for transparency.


In [ ]:
adata.var[adata.var.index.str.contains("CDKN2A")]

In [ ]:
adata.var[adata.var.index.str.contains("CDKN2A-")]

In [ ]:
cdkn2a = adata[:, adata.var.index.str.contains("CDKN2A-")]
print(f"Number of CDKN2A isoforms: {cdkn2a.n_vars}")
print(f"Cells expressing any CDKN2A isoform: {int((cdkn2a.X.sum(axis=1) > 0).sum())}")

In [ ]:
def classify_cdkn2a_isoforms(adata, classification):
    cd = adata[:, adata.var.index.str.contains(r"^CDKN2A($|-)", regex=True)]

    rows = []
    for name in cd.var.index:
        pb_id = cd.var.loc[name, "gene_ids"].split(":")[0]
        match = classification[classification["isoform"] == pb_id]
        if len(match) == 0:
            continue
        transcript = match["associated_transcript"].values[0]
        base = str(transcript).split(".")[0]
        protein = "p14" if base == P14_ENST else "p16" if base == P16_ENST else "other"
        rows.append({
            "isoform": name,
            "pb_id": pb_id,
            "transcript": transcript,
            "protein": protein,
            "structural_category": match["structural_category"].values[0],
            "diff_to_TSS": match["diff_to_TSS"].values[0],
            "diff_to_TTS": match["diff_to_TTS"].values[0],
            "total_reads": float(cd[:, name].X.sum()),
            "cells_expressing": int((cd[:, name].X > 0).sum()),
        })
    return pd.DataFrame(rows).sort_values("total_reads", ascending=False)

cdkn2a_map = classify_cdkn2a_isoforms(adata, classification)

# rebuild p14_var_names / p16_var_names from this corrected mapping, so every
# downstream cell (feature plots, heatmap, detection rate) uses the fixed list
p14_var_names = cdkn2a_map.loc[cdkn2a_map["protein"] == "p14", "isoform"].tolist()
p16_var_names = cdkn2a_map.loc[cdkn2a_map["protein"] == "p16", "isoform"].tolist()

print(f"p14 isoforms (corrected): {p14_var_names}")
print(f"p16 isoforms (corrected): {p16_var_names}")
cdkn2a_map

## 3. Quality control

In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(adata.obs["total_counts"], bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Total counts per cell"); axes[0].set_title("Count depth")
axes[1].hist(adata.obs["n_genes_by_counts"], bins=100, color="mediumseagreen", edgecolor="none")
axes[1].set_xlabel("Isoforms detected per cell"); axes[1].set_title("Isoforms per cell")
axes[2].hist(adata.obs["pct_counts_mt"], bins=100, color="salmon", edgecolor="none")
axes[2].set_xlabel("% mitochondrial counts"); axes[2].set_title("Mitochondrial %")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{DATASET_NAME}_qc_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"], s=3, alpha=0.5, color="grey")
ax.set_xlabel("total_counts")
ax.set_ylabel("n_genes_by_counts")
plt.tight_layout()
plt.show()

In [ ]:
#check how many cells express p14/p16 isoforms after filtering

def classify_cdkn2a_isoforms(adata, classification):
    # ^CDKN2A($|-) catches BOTH the unsuffixed entry AND the scanpy-deduplicated
    # ones ("CDKN2A-1", "CDKN2A-2", ...). "CDKN2A-" alone misses the first
    # isoform scanpy encountered - a real isoform, not a naming artifact.
    cd = adata[:, adata.var.index.str.contains(r"^CDKN2A($|-)", regex=True)]

    rows = []
    for name in cd.var.index:
        pb_id = cd.var.loc[name, "gene_ids"].split(":")[0]
        match = classification[classification["isoform"] == pb_id]
        if len(match) == 0:
            continue
        transcript = match["associated_transcript"].values[0]
        base = str(transcript).split(".")[0]
        protein = "p14" if base == P14_ENST else "p16" if base == P16_ENST else "other"
        rows.append({
            "isoform": name,
            "pb_id": pb_id,
            "transcript": transcript,
            "protein": protein,
            "structural_category": match["structural_category"].values[0],
            "diff_to_TSS": match["diff_to_TSS"].values[0],
            "diff_to_TTS": match["diff_to_TTS"].values[0],
            "total_reads": float(cd[:, name].X.sum()),
            "cells_expressing": int((cd[:, name].X > 0).sum()),
        })
    return pd.DataFrame(rows).sort_values("total_reads", ascending=False)

cdkn2a_map = classify_cdkn2a_isoforms(adata, classification)

# rebuild p14_var_names / p16_var_names from this corrected mapping, so every
# downstream cell (feature plots, heatmap, detection rate) uses the fixed list
p14_var_names = cdkn2a_map.loc[cdkn2a_map["protein"] == "p14", "isoform"].tolist()
p16_var_names = cdkn2a_map.loc[cdkn2a_map["protein"] == "p16", "isoform"].tolist()

print(f"p14 isoforms (corrected): {p14_var_names}")
print(f"p16 isoforms (corrected): {p16_var_names}")
cdkn2a_map

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"Highly variable isoforms: {int(adata.var.highly_variable.sum())}")

adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

## 5. Clustering, UMAP, and cell-type annotation (CellTypist — provisional)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15)   # set n_pcs from the elbow plot above
sc.tl.umap(adata, random_state=42)                  # fixed seed - consistent orientation on reruns
sc.tl.leiden(adata, resolution=1.0)            # matches your earlier reference UMAP's granularity

print(f"Clusters found: {adata.obs['leiden'].nunique()}")
sc.pl.umap(adata, color="leiden", palette=CATEGORICAL_PALETTE, legend_loc="on data", frameon=False)

In [ ]:
adata_raw_reload = sc.read_10x_mtx(mtx_path, var_names="gene_symbols", cache=False)
adata_raw_reload.var_names_make_unique()
adata.layers["counts"] = adata_raw_reload[adata.obs_names, adata.var_names].X.copy()

adata_ct = adata.copy()
adata_ct.X = adata_ct.layers["counts"].copy()
sc.pp.normalize_total(adata_ct, target_sum=1e4)
sc.pp.log1p(adata_ct)
print(adata_ct.X.max())

In [ ]:
import celltypist
from celltypist import models

models.download_models(model=["Immune_All_High.pkl"])

predictions = celltypist.annotate(adata_ct, model="Immune_All_High.pkl", majority_voting=True)
adata.obs["cell_type"] = predictions.predicted_labels["majority_voting"].values

In [ ]:
# gene-level matrix - used ONLY for CellTypist annotation, nowhere else
GENE_MATRIX_DIR = DATA_DIR / "gene matrix"

adata_genes = sc.read_10x_mtx(GENE_MATRIX_DIR, var_names="gene_symbols", cache=False)
adata_genes.var_names_make_unique()

# match to the SAME cells that survived your isoform-level QC filtering
adata_genes = adata_genes[adata_genes.obs_names.isin(adata.obs_names)].copy()
adata_genes = adata_genes[adata.obs_names, :].copy()   # align order to match adata exactly

print(f"Cells in isoform matrix: {adata.n_obs}")
print(f"Cells matched in gene matrix: {adata_genes.n_obs}")

sc.pp.normalize_total(adata_genes, target_sum=1e4)
sc.pp.log1p(adata_genes)

predictions = celltypist.annotate(adata_genes, model='Immune_All_High.pkl', majority_voting=True,
                                    over_clustering=adata.obs["leiden"].values)
adata.obs["cell_type"] = predictions.predicted_labels["majority_voting"].values

print(f"\nCell types identified: {adata.obs['cell_type'].nunique()}")
print(adata.obs["cell_type"].value_counts())

fig, ax = plt.subplots(figsize=(7, 6))
sc.pl.umap(adata, color="cell_type", palette=CATEGORICAL_PALETTE,
           legend_loc="right margin", frameon=False, ax=ax, show=False,
           title="Cell types (CellTypist, real gene matrix, provisional)")
plt.tight_layout()
plt.show()

In [ ]:
def add_gene_total(adata, gene):
    # Sum expression across all isoforms of a gene (base + '-N'), read from .raw.
    names = adata.raw.var_names
    mask = (names == gene) | names.str.startswith(gene + "-")
    if mask.sum() == 0:
        print(f"  {gene}: not found")
        return False
    sub = adata.raw[:, names[mask]].X
    total = np.asarray(sub.sum(axis=1)).flatten()
    adata.obs[f"{gene}_total"] = total
    return True

marker_genes = [
    "CD3D", "CD3E", "TRAC",             # pan-T cell
    "CD8A", "CD8B",                     # CD8 T
    "GNLY", "NKG7",                     # NK cell
    "MS4A1", "CD79A",                   # B cell
    "MZB1", "JCHAIN",                   # plasma cell
    "CD68", "LYZ", "CD14",              # macrophage / monocyte
    "CA9", "NDUFA4L2",                  # ccRCC tumour epithelium (VHL-HIF axis)
    "CST3",                             # ambient/highly-expressed transcript (contamination check)
    "EPCAM", "KRT8", "KRT18",           # general epithelial
]

present = [g for g in marker_genes if add_gene_total(adata, g)]

# clear grey->colour ramp instead of viridis, per this notebook's convention
marker_cmap = clear_cmap("#C0392B")
sc.pl.umap(adata, color=[f"{g}_total" for g in present], ncols=4, cmap=marker_cmap, frameon=False)

In [ ]:
adata_genes = sc.read_10x_mtx(GENE_MATRIX_DIR, var_names="gene_symbols", cache=False)
adata_genes.var_names_make_unique()
print(adata_genes.var_names[:20].tolist())
print('CA9' in adata_genes.var_names, 'PTPRC' in adata_genes.var_names)

In [ ]:
renal_markers = ['CA9', 'NDUFA4L2', 'PAX8', 'LRP2', 'CUBN']
immune_markers = ['PTPRC', 'CD3D', 'CD68', 'CD79A', 'MZB1']
present_renal = [g for g in renal_markers if g in adata_genes.var_names]
present_immune = [g for g in immune_markers if g in adata_genes.var_names]
print("Renal:", present_renal)
print("Immune:", present_immune)

In [ ]:
present_markers = present_renal + present_immune

sc.pl.umap(adata_genes, color=present_markers, ncols=3,
           cmap=mcolors.LinearSegmentedColormap.from_list("mk", ["#E6E6E6", "#7A1F3D"]),
           frameon=False)

In [ ]:
adata_genes = adata_genes[adata.obs_names].copy()
adata_genes.obsm['X_umap'] = adata.obsm['X_umap']
adata_genes.obs['leiden'] = adata.obs['leiden'].values

sc.pl.umap(adata_genes, color=present_renal + present_immune, ncols=3,
           cmap=mcolors.LinearSegmentedColormap.from_list("mk", ["#E6E6E6", "#7A1F3D"]),
           frameon=False)

In [ ]:
for g in present_renal + present_immune:
    match = adata_genes.var_names[adata_genes.var_names == g]
    print(g, "->", len(match), "match(es)")

print("adata_genes cells:", adata_genes.n_obs, "| adata cells:", adata.n_obs)
print("duplicated var_names:", adata_genes.var_names.duplicated().sum())

In [ ]:
renal_markers = ['CA9', 'NDUFA4L2', 'PAX8', 'LRP2', 'CUBN']
immune_markers = ['PTPRC', 'CD3D', 'CD68', 'CD79A', 'MZB1']

present_renal = [g for g in renal_markers if g in adata_ct.var_names]
present_immune = [g for g in immune_markers if g in adata_ct.var_names]
print("Renal markers found:", present_renal)
print("Immune markers found:", present_immune)

sc.pl.umap(adata_ct, color=present_renal + present_immune, ncols=3,
           cmap=mcolors.LinearSegmentedColormap.from_list("mk", ["#E6E6E6", "#7A1F3D"]))

In [ ]:
adata_ct.obs['leiden'] = adata.obs['leiden'].values  # bring cluster labels across
marker_means = adata_ct.obs.groupby('leiden')[present_renal + present_immune].mean() \
    if all(m in adata_ct.obs.columns for m in present_renal+present_immune) else None

# proper way: pull expression via adata_ct[:, markers].X, not adata_ct.obs
import pandas as pd
expr = pd.DataFrame(adata_ct[:, present_renal + present_immune].X.toarray(),
                     columns=present_renal + present_immune, index=adata_ct.obs_names)
expr['leiden'] = adata.obs['leiden'].values
print(expr.groupby('leiden').mean())

## 6. p14 / p16 feature plots (clear palette, not viridis)

In [ ]:
def plot_isoform_feature(adata, isoform_names, color_hex, cmap, title, ax=None):
    """Feature plot with a clear grey (zero) -> colour (max) scale, zero cells
    drawn first so sparse expressing cells aren't visually buried."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    idx = [adata.raw.var_names.get_loc(i) for i in isoform_names if i in adata.raw.var_names]
    if not idx:
        ax.set_title(f"{title} (not detected)")
        ax.axis("off")
        return ax
    expr = np.asarray(adata.raw.X[:, idx].sum(axis=1)).ravel()
    coords = adata.obsm["X_umap"]
    is_zero = expr == 0
    ax.scatter(coords[is_zero, 0], coords[is_zero, 1], c="#E6E6E6", s=8, linewidths=0)
    sca = ax.scatter(coords[~is_zero, 0], coords[~is_zero, 1], c=expr[~is_zero],
                      cmap=cmap, vmin=0, vmax=max(expr.max(), 1), s=30,
                      edgecolors="black", linewidths=0.4)
    if (~is_zero).sum() > 0:
        plt.colorbar(sca, ax=ax, shrink=0.6, label="counts")
    ax.set_title(f"{title}  (n={int((~is_zero).sum())} cells, max={int(expr.max()) if len(expr) else 0})")
    ax.set_xticks([]); ax.set_yticks([])
    return ax

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_isoform_feature(adata, surviving_p14, P14_COLOR, CMAP_P14, "p14ARF", ax=axes[0])
plot_isoform_feature(adata, surviving_p16, P16_COLOR, CMAP_P16, "p16INK4a", ax=axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{DATASET_NAME}_p14_p16_umap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#---- Fixed colour mapping: same cell type = same colour across every dataset ----#
CELL_TYPE_COLORS = {
    "B cells": "#4C72B0",
    "ILC": "#DD8452",
    "Monocytes": "#55A868",
    "Plasma cells": "#C44E52",
    "T cells": "#8172B2",
    "Epithelial cells": "#937860",
    "Macrophages": "#DA8BC3",
    "Myelocytes": "#8C8C8C",
    "DC": "#CCB974"
}

adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
adata.uns["cell_type_colors"] = [
    CELL_TYPE_COLORS[ct] for ct in adata.obs["cell_type"].cat.categories
]

# ---- p14ARF/p16INK4A overlay on annotated UMAP ----
fig, ax = plt.subplots(figsize=(8, 8))

sc.pl.umap(adata, color="cell_type", legend_loc="right margin", frameon=False,
           ax=ax, show=False,
           title="ccRCC Tumour — p14ARF and p16INK4A expression across annotated cell types",
           alpha=0.45, size=35)
coords = adata.obsm["X_umap"]

p14_idx = [adata.raw.var_names.get_loc(i) for i in p14_var_names if i in adata.raw.var_names]
p16_idx = [adata.raw.var_names.get_loc(i) for i in p16_var_names if i in adata.raw.var_names]
p14_expr = np.asarray(adata.raw.X[:, p14_idx].sum(axis=1)).ravel() if p14_idx else np.zeros(adata.n_obs)
p16_expr = np.asarray(adata.raw.X[:, p16_idx].sum(axis=1)).ravel() if p16_idx else np.zeros(adata.n_obs)

p14_cells = p14_expr > 0
p16_cells = p16_expr > 0

cell_type_handles, cell_type_labels = ax.get_legend_handles_labels()

p14_scatter = ax.scatter(coords[p14_cells, 0], coords[p14_cells, 1], marker="^", s=70,
           facecolors=P14_COLOR, edgecolors="black", linewidths=0.8,
           label=f"p14ARF (n={p14_cells.sum()})")

p16_scatter = ax.scatter(coords[p16_cells, 0], coords[p16_cells, 1], marker="v", s=70,
           facecolors=P16_COLOR, edgecolors="black", linewidths=0.8,
           label=f"p16INK4A (n={p16_cells.sum()})")

all_handles = cell_type_handles + [p14_scatter, p16_scatter]
all_labels = cell_type_labels + [f"p14ARF (n={p14_cells.sum()})", f"p16INK4A (n={p16_cells.sum()})"]
ax.legend(all_handles, all_labels, loc="upper right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{DATASET_NAME}_celltype_p14_p16_overlay.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. p14 / p16 × cell-type heatmap

In [ ]:
def isoform_expr_series(adata, isoform_names):
    idx = [adata.raw.var_names.get_loc(i) for i in isoform_names if i in adata.raw.var_names]
    if not idx:
        return pd.Series(0, index=adata.obs_names)
    return pd.Series(np.asarray(adata.raw.X[:, idx].sum(axis=1)).ravel(), index=adata.obs_names)

adata.obs["p14_expr"] = isoform_expr_series(adata, p14_var_names)
adata.obs["p16_expr"] = isoform_expr_series(adata, p16_var_names)

heatmap_data = pd.DataFrame({
    "p14ARF": adata.obs.groupby("cell_type")["p14_expr"].mean(),
    "p16INK4a": adata.obs.groupby("cell_type")["p16_expr"].mean(),
}).T

fig, ax = plt.subplots(figsize=(max(6, 0.55 * heatmap_data.shape[1]), 3))
im = ax.imshow(heatmap_data.values, cmap=mcolors.LinearSegmentedColormap.from_list("hm", ["#E6E6E6", "#7A1F3D"]), aspect="auto")
ax.set_xticks(range(heatmap_data.shape[1])); ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right")
ax.set_yticks([0, 1]); ax.set_yticklabels(["p14ARF", "p16INK4a"])
vmax = heatmap_data.values.max() if heatmap_data.values.max() > 0 else 1
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val > vmax/2 else "black", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.7, label="mean expression")
ax.set_title(f"{DATASET_NAME}: p14/p16 mean expression by cell type")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{DATASET_NAME}_p14p16_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

heatmap_data

**What this does:** mean expression per cell type, in the structure comparable to
Elena's short-read figure — same summary statistic, cell-type columns, gene rows.

**Caveat, worth raising with Elena directly:** at ~1 read/cell typical for this
locus, mean expression is heavily influenced by one or two outlier cells within a
cell type, especially in small clusters. Pair this figure with the detection-rate
chart below (Section 8), which is a more robust statistic at this sparsity — don't
present the heatmap alone as the full picture.

## 8. Detection rate by cell type

In [ ]:
detection = adata.obs.groupby("cell_type").agg(
    pct_p14=("p14_expr", lambda x: 100 * (x > 0).mean()),
    pct_p16=("p16_expr", lambda x: 100 * (x > 0).mean()),
    n_cells=("p14_expr", "size"),
).round(2)
print(detection)

fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(detection)), 4))
detection[["pct_p14", "pct_p16"]].plot(kind="bar", ax=ax, color=[P14_COLOR, P16_COLOR])
ax.set_ylabel("% cells expressing"); ax.set_xlabel("")
ax.legend(["p14ARF", "p16INK4a"])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{DATASET_NAME}_detection_rate.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Save summary for cross-dataset comparison

In [ ]:
summary = {
    "dataset": DATASET_NAME,
    "n_cells_final": int(adata.n_obs),
    "n_isoforms_final": int(adata.n_vars),
    "p14_isoform_ids": p14_var_names,
    "p16_isoform_ids": p16_var_names,
    "pct_cells_expressing_p14": float((adata.obs["p14_expr"] > 0).mean() * 100),
    "pct_cells_expressing_p16": float((adata.obs["p16_expr"] > 0).mean() * 100),
    "n_cell_types_annotated": int(adata.obs["cell_type"].nunique()),
}
for k, v in summary.items():
    print(f"{k}: {v}")

pd.Series(summary).to_json(OUTPUT_DIR / f"{DATASET_NAME}_summary.json")
detection.to_csv(OUTPUT_DIR / f"{DATASET_NAME}_detection_by_celltype.csv")
heatmap_data.to_csv(OUTPUT_DIR / f"{DATASET_NAME}_heatmap_by_celltype.csv")
adata.write(OUTPUT_DIR / f"{DATASET_NAME}_processed.h5ad")
print(f"\nSaved to {OUTPUT_DIR}")

In [ ]:
print(f"=== {DATASET_NAME} ===")
print(f"Total isoforms in adata: {adata.n_vars}")
print(f"p14ARF total reads: {adata.obs['p14_expr'].sum()}")
print(f"p16INK4A total reads: {adata.obs['p16_expr'].sum()}")
ratio = adata.obs['p14_expr'].sum() / adata.obs['p16_expr'].sum() if adata.obs['p16_expr'].sum() > 0 else "undefined"
print(f"p14/p16 ratio: {ratio}")
print(f"n_cells: {adata.n_obs}")
print(f"\nCell type counts:")
print(adata.obs['cell_type'].value_counts())
print(f"\nPercent of total assigned (non-null cell_type): {100 * adata.obs['cell_type'].notna().mean():.1f}%")